In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [ ]:
# Q21: Churn prediction feature dataset using Pandas feature engineering
# Take the raw dataset and transform it into a clean numerical table ready for a machine learning model to use.

# 1. Copied numerical columns as-is (tenure, charges, CLV)
# 2. Converted Yes/No columns to 1/0 (binary encoding)
# 3. Split Contract into separate columns (one-hot encoding)
# 4. Added risk_score from Q10 as an engineered feature
# 5. Added churn as the target variable (what the model predicts)

features = pd.DataFrame()

features['customerID']= df['customerID']

# numerical features
features['tenure'] = df['tenure']
features['monthly_charges']= df['MonthlyCharges']
features['total_charges'] = df['TotalCharges']
features['clv'] = df['CLV']

# binary encoded features
features['is_senior'] = df['SeniorCitizen']
features['has_partner'] = (df['Partner'] == 'Yes').astype(int)# Yes=1, No=0
features['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)
features['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)
features['has_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)
features['has_techsupport'] = (df['TechSupport'] == 'Yes').astype(int)
features['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)

# contract encoded
features['contract_monthly'] = (df['Contract'] == 'Month-to-month').astype(int)
features['contract_yearly'] = (df['Contract'] == 'One year').astype(int)

# risk score from Q10
features['risk_score'] = features.apply(lambda row: 
    (3 if df.loc[row.name, 'Contract'] == 'Month-to-month' else 1 if df.loc[row.name, 'Contract'] == 'One year' else 0) +
    (3 if row['tenure'] <= 12 else 2 if row['tenure'] <= 24 else 1 if row['tenure'] <= 48 else 0) +
    (2 if df.loc[row.name, 'TechSupport'] == 'No' else 0) +
    (2 if df.loc[row.name, 'OnlineSecurity'] == 'No' else 0), axis=1)

# target variable
features['churn'] = (df['Churn'] == 'Yes').astype(int) # Yes=1, No=0

features.head()


,customerID,tenure,monthly_charges,total_charges,clv,is_senior,has_partner,has_dependents,has_phone,has_security,has_techsupport,is_paperless,contract_monthly,contract_yearly,risk_score,churn
0,7590-VHVEG,1,29.85,29.85,29.85,0,1,0,0,0,0,1,1,0,10,0
1,5575-GNVDE,34,56.95,1889.50,1936.30,0,0,0,1,1,0,0,0,1,4,0
2,3668-QPYBK,2,53.85,108.15,107.70,0,0,0,1,1,0,1,1,0,8,1
3,7795-CFOCW,45,42.30,1840.75,1903.50,0,0,0,0,1,1,0,0,1,2,0
4,9237-HQITU,2,70.70,151.65,141.40,0,0,0,1,0,0,1,1,0,10,1


In [6]:
# Q23: Retention dashboard dataset with retention KPIs
retention = {}

# How many customers total? → count all rows → 7,043
retention['total_customers'] = len(df)

# How many left? → count where Churn = Yes → 1,869
retention['churned_customers'] = (df['Churn'] == 'Yes').sum()

# How many stayed? → count where Churn = No → 5,174
retention['retained_customers'] = (df['Churn'] == 'No').sum()

# What % left? → 1,869 / 7,043 * 100 → 26.54%
retention['churn_rate'] = round((df['Churn'] == 'Yes').sum() / len(df) * 100, 2)

# What % stayed? → 5,174 / 7,043 * 100 → 73.46%
retention['retention_rate'] = round((df['Churn'] == 'No').sum() / len(df) * 100, 2)

# How much money do we lose monthly? → add up charges of churned customers → $139,130
retention['monthly_revenue'] = round(df['MonthlyCharges'].sum(), 2)
retention['monthly_revenue_lost'] = round(df[df['Churn'] == 'Yes']['MonthlyCharges'].sum(), 2)

# How much is that per year? → multiply by 12 → $1,669,570
retention['annual_revenue_lost'] = round(retention['monthly_revenue_lost'] * 12, 2)

# Are retained customers worth more? → compare their average CLV → yes, $2,549 vs $1,531
retention['avg_clv_retained'] = round(df[df['Churn'] == 'No']['CLV'].mean(), 2)
retention['avg_clv_churned'] = round(df[df['Churn'] == 'Yes']['CLV'].mean(), 2)

# print dashboard
for key, value in retention.items():
    print(f'{key:35s}: {value}')


total_customers                    : 7043
churned_customers                  : 1869
retained_customers                 : 5174
churn_rate                         : 26.54
retention_rate                     : 73.46
monthly_revenue                    : 456116.6
monthly_revenue_lost               : 139130.85
annual_revenue_lost                : 1669570.2
avg_clv_retained                   : 2549.77
avg_clv_churned                    : 1531.61


In [ ]:
# Q24: Rule-based churn detection engine
# assigns risk levels: Low, Medium, High, Critical
def churn_risk_level(row):
    score = 0

    # Check their contract. Month-to-month = most dangerous, add 3. One year = some risk, add 1. Two year = safe, add nothing.
    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1

    # Check how long they've been a customer. Newer = more risk = more points.
    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1

    # Two separate if statements — both can add points at the same time. A customer with neither service gets +4.
    if row['OnlineSecurity'] == 'No':
        score += 2
    if row['TechSupport'] == 'No':
        score += 2

    # payment risk
    if row['PaymentMethod'] == 'Electronic check':
        score += 1

    # senior citizen risk
    if row['SeniorCitizen'] == 1:
        score += 1

    # Based on total score, return a label. Maximum possible score = 12 → Critical.
    if score >= 9:
        return 'Critical'
    elif score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'

df['churn_risk'] = df.apply(churn_risk_level, axis=1)

# summarize results
df.groupby('churn_risk').agg(
    customer_count = ('customerID', 'count'),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index().sort_values('churn_rate', ascending=False)


,churn_risk,customer_count,churn_rate
0,Critical,2105,56.72
1,High,1764,27.61
3,Medium,1377,9.37
2,Low,1797,3.28
